# Notebook 04: Text Preprocessing and Document Chunking

## Project: Advance AI Research Assistant

### Objective

This notebook prepares extracted scientific PDF documents
for the Retrieval-Augmented Generation (RAG) pipeline.

The main tasks are:

1. Load extracted PDF documents.
2. Normalize and clean extracted text.
3. Preserve document and page metadata.
4. Split documents into manageable, overlapping chunks.
5. Prepare structured chunks for embedding and vector storage.

### Expected Output

A collection of clean text chunks containing:

- Document ID
- Document filename
- Page number
- Chunk ID
- Chunk text
- Character and word counts

## 2. Environment and Library Setup

In [1]:
import sys
import re
import json
import unicodedata
from pathlib import Path

import fitz
import pandas as pd
import numpy as np

# Project configuration
PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

DATA_DIR = PROJECT_ROOT / "data"
RAW_PDF_DIR = DATA_DIR / "raw_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Raw PDFs:", RAW_PDF_DIR)
print("Processed data:", PROCESSED_DIR)

assert PROJECT_ROOT.exists(), "Project directory not found."
assert RAW_PDF_DIR.exists(), "PDF directory not found."

Python: 3.11.17 (main, Oct 13 2026, 16:45:47) [GCC 15.3.0]
Project root: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
Raw PDFs: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/raw_pdfs
Processed data: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/processed


## 3. Load Extracted PDF Documents

Load all available PDF documents using PyMuPDF.

Each page will be stored separately to preserve page-level
information for later source citations.

In [2]:
def load_pdf_documents(pdf_directory: Path) -> list:
    """Extract all PDF documents page by page using PyMuPDF."""

    documents = []

    pdf_files = sorted(pdf_directory.glob("*.pdf"))

    for pdf_path in pdf_files:

        try:
            with fitz.open(pdf_path) as pdf:

                for page_index, page in enumerate(pdf):

                    text = page.get_text("text", sort=True)

                    documents.append({
                        "document_id": pdf_path.stem,
                        "filename": pdf_path.name,
                        "page_number": page_index + 1,
                        "text": text
                    })

            print(f"Loaded: {pdf_path.name}")

        except Exception as error:
            print(f"Failed: {pdf_path.name} - {error}")

    return documents


documents = load_pdf_documents(RAW_PDF_DIR)

print("\nTotal extracted pages:", len(documents))
print("Total documents:", len(set(d["document_id"] for d in documents)))

Loaded: 41598_2019_Article_38798.pdf
Loaded: NIPS-2017-attention-is-all-you-need-Paper.pdf
Loaded: s41598-019-52937-5.pdf

Total extracted pages: 39
Total documents: 3


In [20]:
documents_df = pd.DataFrame([
    {
        "filename": d["filename"],
        "page_number": d["page_number"],
        "characters": len(d["text"]),
        "words": len(d["text"].split())
    }
    for d in documents
])

display(documents_df.head(10))

,filename,page_number,characters,words
0,41598_2019_Article_38798.pdf,1,6102,646
1,41598_2019_Article_38798.pdf,2,8878,1068
2,41598_2019_Article_38798.pdf,3,6962,866
3,41598_2019_Article_38798.pdf,4,1825,217
4,41598_2019_Article_38798.pdf,5,8823,1081
5,41598_2019_Article_38798.pdf,6,5781,750
6,41598_2019_Article_38798.pdf,7,5200,630
7,41598_2019_Article_38798.pdf,8,3139,367
8,41598_2019_Article_38798.pdf,9,3012,365
9,41598_2019_Article_38798.pdf,10,2680,324


## 4. Text Preprocessing

Scientific documents contain formatting artifacts such as
ligatures, line breaks, and inconsistent whitespace.

This section normalizes extracted text while preserving
important scientific information, including Greek letters,
mathematical symbols, and technical terminology.

In [7]:
def clean_text(text: str) -> str:
    """Clean and normalize extracted PDF text."""

    # Normalize Unicode characters and ligatures.
    text = unicodedata.normalize("NFKC", text)

    # Remove soft hyphens.
    text = text.replace("\u00ad", "")

    # Join words split across lines, e.g. transduc-\ntion.
    text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", text)

    # Normalize line endings.
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Replace line breaks and tabs with spaces.
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing whitespace.
    text = text.strip()

    return text

In [21]:
# Testing on the Attention paper
sample_text = documents[0]["text"]
cleaned_sample = clean_text(sample_text)

print("ORIGINAL TEXT:")
print(sample_text[:800])

print("\nCLEANED TEXT:")
print(cleaned_sample[:800])

print("\nOriginal characters:", len(sample_text))
print("Cleaned characters:", len(cleaned_sample))

ORIGINAL TEXT:
                                                                             www.nature.com/scientificreports




       OPEN Cell Identity Codes: Understanding
              Cell Identity from Gene Expression
              Profiles using Deep Neural
Received: 5 September 2017Accepted: 10 January 2019    Networks
Published: xx xx xxxx                              Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1,
                           Seyed Abolfazl Motahari1, Ali Sharifi-Zarchi      1 & Hamidreza Chitsaz3

                                 Understanding cell identity is an important task in many biomedical areas. Expression patterns of
                                         specific marker genes have been used to characterize some limited cell types, but e

CLEANED TEXT:
www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted

In [12]:
for document in documents:
    document["cleaned_text"] = clean_text(document["text"])

print("Text preprocessing completed.")

print(
    "Pages processed:",
    sum(bool(d["cleaned_text"]) for d in documents),
    "/",
    len(documents)
)

Text preprocessing completed.
Pages processed: 39 / 39


In [13]:
# 5. Check preprocessing results
preprocessing_report = pd.DataFrame([
    {
        "filename": d["filename"],
        "page_number": d["page_number"],
        "original_characters": len(d["text"]),
        "cleaned_characters": len(d["cleaned_text"]),
        "original_words": len(d["text"].split()),
        "cleaned_words": len(d["cleaned_text"].split()),
        "is_empty": not bool(d["cleaned_text"])
    }
    for d in documents
])

display(preprocessing_report.head(10))

print("\nEmpty pages:", preprocessing_report["is_empty"].sum())
print(
    "Total cleaned characters:",
    preprocessing_report["cleaned_characters"].sum()
)

,filename,page_number,original_characters,cleaned_characters,original_words,cleaned_words,is_empty
0,41598_2019_Article_38798.pdf,1,6102,4402,646,643,False
1,41598_2019_Article_38798.pdf,2,8878,6855,1068,1061,False
2,41598_2019_Article_38798.pdf,3,6962,5152,866,864,False
3,41598_2019_Article_38798.pdf,4,1825,1402,217,216,False
4,41598_2019_Article_38798.pdf,5,8823,6791,1081,1072,False
5,41598_2019_Article_38798.pdf,6,5781,4467,750,744,False
6,41598_2019_Article_38798.pdf,7,5200,4020,630,624,False
7,41598_2019_Article_38798.pdf,8,3139,2438,367,366,False
8,41598_2019_Article_38798.pdf,9,3012,2321,365,362,False
9,41598_2019_Article_38798.pdf,10,2680,2013,324,322,False



Empty pages: 0
Total cleaned characters: 150143


## 6. Document Chunking

Large documents are divided into smaller text segments called
chunks.

Each chunk retains its document ID, filename, and original
PDF page number.

We use overlapping chunks to reduce the risk of losing
important context at chunk boundaries.

Initial configuration:
- Chunk size: 1,200 characters
- Chunk overlap: 150 characters

In [14]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150


def split_text_into_chunks(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP
) -> list[str]:
    """Split text into overlapping chunks at word boundaries."""

    if not text.strip():
        return []

    if overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk size.")

    chunks = []
    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        # Prefer ending at a word boundary.
        if end < len(text):
            boundary = text.rfind(" ", start + chunk_size // 2, end)

            if boundary > start:
                end = boundary

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

In [15]:
# 7. Create structured document chunks
all_chunks = []

for document in documents:

    page_chunks = split_text_into_chunks(
        document["cleaned_text"]
    )

    for chunk_index, chunk_text in enumerate(page_chunks):

        chunk_id = (
            f"{document['document_id']}"
            f"_p{document['page_number']}"
            f"_c{chunk_index + 1}"
        )

        all_chunks.append({
            "chunk_id": chunk_id,
            "document_id": document["document_id"],
            "filename": document["filename"],
            "page_number": document["page_number"],
            "chunk_index": chunk_index + 1,
            "text": chunk_text,
            "character_count": len(chunk_text),
            "word_count": len(chunk_text.split())
        })

print("Total chunks created:", len(all_chunks))

Total chunks created: 156


In [16]:
# 8. Inspect and validate chunks
chunks_df = pd.DataFrame(all_chunks)

display(chunks_df.head(10))

print("Total chunks:", len(chunks_df))
print("Unique chunk IDs:", chunks_df["chunk_id"].nunique())
print("Empty chunks:", (chunks_df["text"].str.strip() == "").sum())
print("Average chunk characters:", round(
    chunks_df["character_count"].mean(), 2
))
print("Maximum chunk characters:", chunks_df["character_count"].max())
print("Minimum chunk characters:", chunks_df["character_count"].min())

,chunk_id,document_id,filename,page_number,chunk_index,text,character_count,word_count
0,41598_2019_Article_38798_p1_c1,41598_2019_Article_38798,41598_2019_Article_38798.pdf,1,1,www.nature.com/scientificreports OPEN Cell Ide...,1199,174
1,41598_2019_Article_38798_p1_c2,41598_2019_Article_38798,41598_2019_Article_38798.pdf,1,2,ific structure of deep autoencoders that can e...,1197,193
2,41598_2019_Article_38798_p1_c3,41598_2019_Article_38798,41598_2019_Article_38798.pdf,1,3,"uripotent stem cells (iPSCs), to some specific...",1198,185
3,41598_2019_Article_38798_p1_c4,41598_2019_Article_38798,41598_2019_Article_38798.pdf,1,4,s4. One important possible solution is to reco...,1197,160
4,41598_2019_Article_38798_p1_c5,41598_2019_Article_38798,41598_2019_Article_38798.pdf,1,5,nce and requests for materials should be addre...,211,24
5,41598_2019_Article_38798_p2_c1,41598_2019_Article_38798,41598_2019_Article_38798.pdf,2,1,www.nature.com/scientificreports/ Kullback-Lei...,1197,182
6,41598_2019_Article_38798_p2_c2,41598_2019_Article_38798,41598_2019_Article_38798.pdf,2,2,ubnetworks are identified using Gene Set Enric...,1198,185
7,41598_2019_Article_38798_p2_c3,41598_2019_Article_38798,41598_2019_Article_38798.pdf,2,3,"ypes. As a result, a simple classifier might h...",1199,176
8,41598_2019_Article_38798_p2_c4,41598_2019_Article_38798,41598_2019_Article_38798.pdf,2,4,"ns than the dimensionality of the data, hence ...",1196,187
9,41598_2019_Article_38798_p2_c5,41598_2019_Article_38798,41598_2019_Article_38798.pdf,2,5,es. Results and Discussion We obtained normali...,1199,188


Total chunks: 156
Unique chunk IDs: 156
Empty chunks: 0
Average chunk characters: 1074.85
Maximum chunk characters: 1199
Minimum chunk characters: 152


In [17]:
assert chunks_df["chunk_id"].is_unique
assert chunks_df["document_id"].notna().all()
assert chunks_df["filename"].notna().all()
assert chunks_df["page_number"].notna().all()
assert chunks_df["text"].str.strip().ne("").all()

print("All chunk validation checks passed.")

All chunk validation checks passed.


## 9. Save the processed chunks
We will save the results in JSONL format. This format is convenient for our later embedding and vector database stages.

In [18]:
CHUNKS_FILE = PROCESSED_DIR / "document_chunks.jsonl"

with CHUNKS_FILE.open("w", encoding="utf-8") as file:

    for chunk in all_chunks:
        file.write(
            json.dumps(chunk, ensure_ascii=False) + "\n"
        )

print("Chunks saved successfully:")
print(CHUNKS_FILE)
print("File size:", CHUNKS_FILE.stat().st_size, "bytes")

Chunks saved successfully:
/home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/processed/document_chunks.jsonl
File size: 204379 bytes


In [19]:
# Verifing the saved file
with CHUNKS_FILE.open("r", encoding="utf-8") as file:
    saved_chunks = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print("Chunks in memory:", len(all_chunks))
print("Chunks saved:", len(saved_chunks))

assert len(saved_chunks) == len(all_chunks)

print("Export verification successful.")

Chunks in memory: 156
Chunks saved: 156
Export verification successful.


## 10. Conclusion

In this notebook, PDF text preprocessing and document chunking
were implemented for Advance AI Research Assistant.

The processing pipeline includes:

1. Loading PDF pages with PyMuPDF.
2. Unicode normalization and text cleaning.
3. Repairing line-break hyphenation.
4. Splitting page text into overlapping chunks.
5. Preserving document IDs and page numbers.
6. Validating and exporting structured chunks.

The processed chunks are stored in:

data/processed/document_chunks.jsonl

These chunks are ready for the embedding and vector database
components of the RAG pipeline.